In [10]:
import pandas as pd

# 1. Load Dataset
file_path = "data/student_exam_performance.csv"
df = pd.read_csv(file_path)

print("=" * 60)
print("INFORMASI AWAL DATASET")
print("=" * 60)

# 2. Cek Jumlah Baris dan Kolom
jumlah_baris, jumlah_kolom = df.shape
print(f"Jumlah Baris : {jumlah_baris:,}")
print(f"Jumlah Kolom : {jumlah_kolom}")
print("-" * 60)

# 3. Cek Missing Value (Nilai Kosong / NaN)
missing_values = df.isnull().sum()
total_missing = missing_values.sum()
print("INFORMASI MISSING VALUE:")
if total_missing == 0:
    print("Tidak ditemukan missing value pada dataset.")
else:
    print(f"Total missing values: {total_missing}")
    kolom_missing = missing_values[missing_values > 0]
    for col, val in kolom_missing.items():
        persen = (val / jumlah_baris) * 100
        print(f"- {col}: {val} ({persen:.2f}%)")
print("-" * 60)

# 4. Cek Data Duplikat
jumlah_duplikat = df.duplicated().sum()
print("INFORMASI DATA DUPLIKAT:")
if jumlah_duplikat == 0:
    print("Tidak ada baris duplikat.")
else:
    print(f"Ditemukan {jumlah_duplikat} baris duplikat.")
print("-" * 60)

# 5. Cek Distribusi Kelas Target
# Kolom target potensial dalam dataset: 'pass_status', 'performance_grade', 'performance_level'
target_columns = ["pass_status", "performance_grade", "performance_level"]

print("DISTRIBUSI KELAS TARGET:")
for target in target_columns:
    if target in df.columns:
        print(f"\n--- Distribusi Target: '{target}' ---")
        counts = df[target].value_counts()
        percentages = df[target].value_counts(normalize=True) * 100
        summary_target = pd.DataFrame({"Jumlah": counts, "Persentase (%)": percentages.round(2)})
        print(summary_target)
print("=" * 60)


INFORMASI AWAL DATASET
Jumlah Baris : 100,000
Jumlah Kolom : 44
------------------------------------------------------------
INFORMASI MISSING VALUE:
Total missing values: 52161
- parent_education: 6526 (6.53%)
- previous_gpa: 7793 (7.79%)
- attendance_percentage: 9863 (9.86%)
- notes_quality: 8407 (8.41%)
- sleep_quality: 6994 (6.99%)
- device_availability: 2965 (2.96%)
- time_management_score: 9613 (9.61%)
------------------------------------------------------------
INFORMASI DATA DUPLIKAT:
Tidak ada baris duplikat.
------------------------------------------------------------
DISTRIBUSI KELAS TARGET:

--- Distribusi Target: 'pass_status' ---
             Jumlah  Persentase (%)
pass_status                        
Pass          77389           77.39
Fail          22611           22.61

--- Distribusi Target: 'performance_grade' ---
                   Jumlah  Persentase (%)
performance_grade                        
D                   34886           34.89
F                   22611     

In [18]:

import pandas as pd

# ==============================================================================
# 1. LOAD DATASET
# ==============================================================================
file_path = "data/student_exam_performance.csv"
df = pd.read_csv(file_path)

# ==============================================================================
# 2. DEFINISI STRUKTUR KOLOM
# ==============================================================================

# a) 13 Fitur Prediktor Terpilih (X) — Fokus kelompok dan nama fitur saja
FITUR_13 = [
    {"Kelompok": "Riwayat akademik", "Fitur": "previous_exam_score"},
    {"Kelompok": "Riwayat akademik", "Fitur": "previous_gpa"},
    {"Kelompok": "Riwayat akademik", "Fitur": "attendance_percentage"},
    {"Kelompok": "Riwayat akademik", "Fitur": "assignment_completion_rate"},
    {"Kelompok": "Kebiasaan belajar", "Fitur": "study_hours_per_day"},
    {"Kelompok": "Kebiasaan belajar", "Fitur": "self_study_hours"},
    {"Kelompok": "Kebiasaan belajar", "Fitur": "practice_tests_completed"},
    {"Kelompok": "Kebiasaan belajar", "Fitur": "exam_preparation_days"},
    {"Kelompok": "Gaya hidup / mental", "Fitur": "sleep_hours"},
    {"Kelompok": "Gaya hidup / mental", "Fitur": "stress_level"},
    {"Kelompok": "Gaya hidup / mental", "Fitur": "exam_anxiety_level"},
    {"Kelompok": "Kondisi ujian", "Fitur": "exam_difficulty"},
    {"Kelompok": "Latar belakang", "Fitur": "family_income"},
]

# b) 1 Kolom Target (y)
TARGET_Y = ["pass_status"]

# c) 5 Kolom Dibuang karena Data Leakage / ID (Tanpa Alasan)
KOLOM_LEAKAGE_ID = [
    {"Kolom": "exam_score", "Kategori": "Data Leakage"},
    {"Kolom": "performance_grade", "Kategori": "Data Leakage"},
    {"Kolom": "performance_level", "Kategori": "Data Leakage"},
    {"Kolom": "questions_correct", "Kategori": "Data Leakage"},
    {"Kolom": "student_id", "Kategori": "Pengenal (ID)"},
]

# d) 25 Kolom Dibuang karena Tidak Relevan (Tanpa Alasan)
LIST_TIDAK_RELEVAN = [
    "age", "gender", "education_level", "school_type", "parent_education", "urban_rural",
    "class_participation", "study_consistency", "study_environment", "study_method",
    "revision_frequency", "notes_quality", "private_tuition", "time_management_score",
    "sleep_quality", "daily_screen_time", "physical_activity_hours", "break_frequency",
    "motivation_level", "internet_access", "device_availability",
    "educational_app_usage", "online_learning_hours", "online_course_hours",
    "questions_attempted"
]

KOLOM_TIDAK_RELEVAN = [
    {"Kolom": col, "Kategori": "Tidak Relevan"}
    for col in LIST_TIDAK_RELEVAN
]

# e) Pengelompokan seluruh 44 kolom dataset awal berdasarkan domain
PENGELOMPOKAN_SEMUA_KOLOM = {
    "Identitas": ["student_id"],
    "Demografi & Latar Belakang": ["age", "gender", "education_level", "school_type", "family_income", "parent_education", "urban_rural"],
    "Riwayat Akademik": ["previous_exam_score", "previous_gpa", "attendance_percentage", "assignment_completion_rate", "class_participation"],
    "Kebiasaan & Metode Belajar": ["study_hours_per_day", "self_study_hours", "private_tuition", "study_consistency", "study_environment", "study_method", "revision_frequency", "notes_quality", "practice_tests_completed", "exam_preparation_days"],
    "Pemanfaatan Teknologi / Online": ["online_learning_hours", "internet_access", "device_availability", "educational_app_usage", "online_course_hours"],
    "Gaya Hidup & Kondisi Mental": ["sleep_hours", "sleep_quality", "daily_screen_time", "physical_activity_hours", "break_frequency", "stress_level", "motivation_level", "time_management_score", "exam_anxiety_level"],
    "Karakteristik & Hasil Ujian": ["exam_difficulty", "questions_attempted", "questions_correct", "exam_score", "performance_grade", "pass_status", "performance_level"]
}

# ==============================================================================
# 3. PENGECEKAN VALIDASI (ASSERT)
# ==============================================================================
list_13_fitur = [item["Fitur"] for item in FITUR_13]
list_leakage = [item["Kolom"] for item in KOLOM_LEAKAGE_ID]
list_tidak_relevan = [item["Kolom"] for item in KOLOM_TIDAK_RELEVAN]

total_terklasifikasi = list_13_fitur + TARGET_Y + list_leakage + list_tidak_relevan

# Memastikan keunikan dan kesesuaian jumlah kolom (44 kolom)
assert len(total_terklasifikasi) == len(set(total_terklasifikasi)), "Ada kolom yang terdaftar ganda!"
assert len(total_terklasifikasi) == 44, f"Total kolom harus 44, ditemukan {len(total_terklasifikasi)}"
assert set(total_terklasifikasi) == set(df.columns), "Kolom yang didefinisikan tidak sesuai dengan dataset CSV!"

# ==============================================================================
# 4. TAMPILAN OUTPUT
# ==============================================================================

# ------------------------------------------------------------------------------
# 1. SEMUA FITUR AWAL (44 KOLOM DIKELOMPOKKAN)
# ------------------------------------------------------------------------------
print("=" * 80)
print("1. DAFTAR SEMUA FITUR AWAL DATASET (44 KOLOM DIKELOMPOKKAN)")
print("=" * 80)

daftar_semua = []
for kelompok, cols in PENGELOMPOKAN_SEMUA_KOLOM.items():
    for col in cols:
        daftar_semua.append({
            "Kelompok Domain": kelompok,
            "Nama Kolom": col,
            "Tipe Data": str(df[col].dtype),
            "Contoh Nilai": str(df[col].dropna().iloc[0])
        })

df_semua_kolom = pd.DataFrame(daftar_semua)
print(df_semua_kolom.to_string(index=False))

# ------------------------------------------------------------------------------
# 2. 13 FITUR PILIHAN (X)
# ------------------------------------------------------------------------------
print("\n" + "=" * 80)
print("2. FITUR PREDIKTOR TERPILIH (13 FITUR X)")
print("=" * 80)
df_fitur_13 = pd.DataFrame(FITUR_13)
print(df_fitur_13.to_string(index=False))

# ------------------------------------------------------------------------------
# 3. TARGET (Y) DAN DISTRIBUSI KELASNYA
# ------------------------------------------------------------------------------
print("\n" + "=" * 80)
print("3. TARGET PREDIKSI (Y): pass_status")
print("=" * 80)
target_nama = TARGET_Y[0]
counts = df[target_nama].value_counts()
percentages = df[target_nama].value_counts(normalize=True) * 100

tabel_target = pd.DataFrame({
    "Kelas": counts.index,
    "Jumlah Baris": counts.values,
    "Persentase (%)": percentages.round(2).values
})
print(tabel_target.to_string(index=False))

# ------------------------------------------------------------------------------
# 4. KOLOM YANG DIBUANG (HANYA KOLOM DAN KATEGORI)
# ------------------------------------------------------------------------------
print("\n" + "=" * 80)
print("4. DAFTAR KOLOM YANG DIBUANG (30 KOLOM: 5 LEAKAGE/ID + 25 TIDAK RELEVAN)")
print("=" * 80)
df_dibuang = pd.DataFrame(KOLOM_LEAKAGE_ID + KOLOM_TIDAK_RELEVAN)
print(df_dibuang.to_string(index=False))

# ------------------------------------------------------------------------------
# 5. RINGKASAN AKHIR PERANAN KOLOM
# ------------------------------------------------------------------------------
print("\n" + "=" * 80)
print("RINGKASAN PERANAN SELURUH KOLOM")
print("=" * 80)
df_ringkasan = pd.DataFrame({
    "Peranan": [
        "Fitur Terpilih (X)",
        "Target Prediksi (y)",
        "Dibuang (Data Leakage & ID)",
        "Dibuang (Tidak Relevan)",
        "Total Kolom Dataset"
    ],
    "Jumlah Kolom": [
        len(list_13_fitur),
        len(TARGET_Y),
        len(list_leakage),
        len(list_tidak_relevan),
        len(total_terklasifikasi)
    ]
})
print(df_ringkasan.to_string(index=False))
print("=" * 80)

1. DAFTAR SEMUA FITUR AWAL DATASET (44 KOLOM DIKELOMPOKKAN)
               Kelompok Domain                 Nama Kolom Tipe Data Contoh Nilai
                     Identitas                 student_id    object   STU_000001
    Demografi & Latar Belakang                        age     int64           20
    Demografi & Latar Belakang                     gender    object       Female
    Demografi & Latar Belakang            education_level    object  High School
    Demografi & Latar Belakang                school_type    object       Public
    Demografi & Latar Belakang              family_income    object       Middle
    Demografi & Latar Belakang           parent_education    object       Master
    Demografi & Latar Belakang                urban_rural    object        Rural
              Riwayat Akademik        previous_exam_score   float64        78.36
              Riwayat Akademik               previous_gpa   float64         2.95
              Riwayat Akademik      attendance_pe

In [9]:
import pandas as pd

df = pd.read_csv("data/student_exam_performance.csv")

fitur = [
    "previous_exam_score", "previous_gpa", "attendance_percentage",
    "assignment_completion_rate", "study_hours_per_day", "self_study_hours",
    "practice_tests_completed", "exam_preparation_days", "sleep_hours",
    "stress_level", "exam_anxiety_level", "exam_difficulty", "family_income",
]
target = "pass_status"

data = df[fitur + [target]].copy()   # hanya 13 fitur + 1 target

assert data.shape[1] == 14           # cek: 13 + 1
data.to_csv("data/data_dipilih.csv", index=False)   # opsional, CSV baru
data.head()

,previous_exam_score,previous_gpa,attendance_percentage,assignment_completion_rate,study_hours_per_day,self_study_hours,practice_tests_completed,exam_preparation_days,sleep_hours,stress_level,exam_anxiety_level,exam_difficulty,family_income,pass_status
0,78.36,2.95,81.49,65.18,4.94,2.63,8,27,6.31,10,9.39,Medium,Middle,Pass
1,73.40,2.89,NaN,76.53,3.42,2.64,10,4,6.81,8,6.31,Easy,Middle,Pass
2,82.76,3.35,100.00,70.39,1.06,0.83,5,28,6.53,9,9.87,Easy,Middle,Pass
3,60.61,2.57,84.91,75.06,7.89,5.51,8,16,6.60,10,9.68,Medium,Middle,Pass
4,74.79,2.84,88.21,61.54,2.63,2.29,1,28,7.83,10,10.00,Medium,High,Pass
